# GulfDealFlow — Script 01: Crunchbase Processor
Normalises a Crunchbase CSV export to the GulfDealFlow schema.

**Before running:** Export your Crunchbase data with these filters:
- HQ Location: UAE, Saudi Arabia, Kuwait, Bahrain, Oman, Qatar
- Funding Type: Seed, Series A, Series B, Series C
- Last Funding Date: 2020 onwards

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install pandas requests beautifulsoup4 lxml -q

In [ ]:
# Create GulfDealFlow folder in Drive
import os
BASE_DIR = '/content/drive/MyDrive/GulfDealFlow'
os.makedirs(BASE_DIR, exist_ok=True)
print(f'✓ Working directory: {BASE_DIR}')

In [ ]:
# Upload your Crunchbase export
from google.colab import files
print('Upload your crunchbase_export.csv now...')
uploaded = files.upload()
for filename in uploaded.keys():
    dest = os.path.join(BASE_DIR, 'crunchbase_export.csv')
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'✓ Saved to: {dest}')

In [ ]:
import pandas as pd
import re
from datetime import datetime

CRUNCHBASE_CSV_PATH = os.path.join(BASE_DIR, 'crunchbase_export.csv')
OUTPUT_PATH         = os.path.join(BASE_DIR, 'crunchbase_cleaned.csv')

COUNTRY_MAP = {
    'United Arab Emirates': 'UAE',
    'Saudi Arabia': 'Saudi Arabia',
    'Kingdom of Saudi Arabia': 'Saudi Arabia',
    'Kuwait': 'Kuwait',
    'Bahrain': 'Bahrain',
    'Oman': 'Oman',
    'Qatar': 'Qatar',
}

STAGE_MAP = {
    'pre_seed': 'Pre-Seed', 'seed': 'Seed', 'series_a': 'Series A',
    'series_b': 'Series B', 'series_c': 'Series C+', 'series_d': 'Series C+',
    'series_e': 'Series C+', 'series_f': 'Series C+', 'growth': 'Growth',
    'venture': 'Undisclosed', 'angel': 'Pre-Seed', 'convertible_note': 'Pre-Seed',
    'corporate_round': 'Growth', 'debt_financing': 'Growth', 'undisclosed': 'Undisclosed',
}

SECTOR_MAP = {
    'financial services': 'Fintech', 'fintech': 'Fintech', 'payments': 'Fintech',
    'insurance': 'Fintech', 'lending': 'Fintech', 'real estate': 'Proptech',
    'proptech': 'Proptech', 'logistics': 'Logistics & Supply Chain',
    'supply chain': 'Logistics & Supply Chain', 'transportation': 'Logistics & Supply Chain',
    'health': 'Healthtech', 'healthcare': 'Healthtech', 'medical': 'Healthtech',
    'edtech': 'Edtech', 'education': 'Edtech', 'e-commerce': 'E-commerce & Retail',
    'retail': 'E-commerce & Retail', 'marketplace': 'E-commerce & Retail',
    'saas': 'SaaS & Enterprise Software', 'enterprise software': 'SaaS & Enterprise Software',
    'artificial intelligence': 'Deep Tech & AI', 'machine learning': 'Deep Tech & AI',
    'deep tech': 'Deep Tech & AI', 'energy': 'Energy & Cleantech',
    'cleantech': 'Energy & Cleantech', 'media': 'Media & Entertainment',
    'food': 'Food & Agritech', 'agriculture': 'Food & Agritech', 'agritech': 'Food & Agritech',
}

def clean_amount(value):
    if pd.isna(value) or str(value).strip() in ['', '--']: return None, False
    cleaned = re.sub(r'[^\d.]', '', str(value))
    if not cleaned: return None, False
    try: return int(float(cleaned)), True
    except: return None, False

def map_sector(cats_str):
    if pd.isna(cats_str): return 'Other'
    for cat in [c.strip().lower() for c in str(cats_str).split(',')]:
        for key, sector in SECTOR_MAP.items():
            if key in cat: return sector
    return 'Other'

def map_stage(ft):
    if pd.isna(ft): return 'Undisclosed'
    return STAGE_MAP.get(str(ft).lower().replace(' ', '_'), 'Undisclosed')

def format_date(d):
    if pd.isna(d): return None
    for fmt in ['%Y-%m-%d', '%m/%d/%Y', '%Y-%m', '%B %Y', '%b %Y']:
        try: return datetime.strptime(str(d).strip(), fmt).strftime('%Y-%m')
        except: continue
    return str(d)[:7]

print(f'Reading: {CRUNCHBASE_CSV_PATH}')
df = pd.read_csv(CRUNCHBASE_CSV_PATH, skiprows=4)
print(f'Raw rows: {len(df)}')
print(f'Columns: {list(df.columns)}\n')

records = []
for _, row in df.iterrows():
    raw_country = str(row.get('Headquarters Location', '')).split(',')[0].strip()
    country = COUNTRY_MAP.get(raw_country)
    if not country: continue
    parts = str(row.get('Headquarters Location', '')).split(',')
    city = parts[1].strip() if len(parts) > 1 else ''
    amount, disclosed = clean_amount(row.get('Funding Amount'))
    if not disclosed: amount, disclosed = clean_amount(row.get('Total Funding Amount'))
    record = {
        'deal_id': '', 'company_name': str(row.get('Organization Name', '')).strip(),
        'country': country, 'city': city, 'date': format_date(row.get('Last Funding Date')),
        'stage': map_stage(row.get('Last Funding Type')),
        'amount_usd': amount if amount else '', 'disclosed': 'TRUE' if disclosed else 'FALSE',
        'sector': map_sector(row.get('Industries')),
        'description': str(row.get('Description', '')).strip()[:200],
        'founded_year': str(row.get('Founded Date', ''))[:4],
        'website': str(row.get('Website', '')).strip(),
        'lead_investor': '', 'co_investors': '', 'investor_types': '',
        'source': 'Crunchbase', 'notes': ''
    }
    if not record['company_name'] or not record['date']: continue
    records.append(record)

out = pd.DataFrame(records)
out.to_csv(OUTPUT_PATH, index=False)
print(f'✓ {len(out)} GCC deals processed')
print(f'✓ Saved: {OUTPUT_PATH}')
print('\nCountry breakdown:')
print(out['country'].value_counts().to_string())
print('\nSector breakdown:')
print(out['sector'].value_counts().to_string())

In [ ]:
# Preview output
out.head(10)